In [ ]:
import numpy as np
from pathlib import Path
import prism

from imagematerials.factory import ModelFactory
from imagematerials.model import RestOf
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.rest_of.preprocessing.main import fit_all_materials_save_corrseponding_input_data

In [ ]:
# REFIT DATA
refit = False

if refit == True:
    fit_all_materials_save_corrseponding_input_data(path_input_data=Path("../data/raw/rest-of"),
                                                    path_input_data_image=Path("../data/raw/image"))

In [ ]:
# Define the complete timeline, including historic tail

YEAR_START  = 1971    # start year of the simulation period
YEAR_END    = 2100    # end year of the calculations
YEAR_OUT    = 2100    # year of output generation = last year of reporting
time_start = 1971
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_OUT, 1)

In [ ]:
scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None)}

In [ ]:

# Define the scenario list
scenario_list = {       
                "base":("SSP2",["base"])
                }

# Define paths
scenario_base_path = Path("..", "data", "raw")
CP_scenario_path = scenario_base_path/ "IMAGE_CircoMod"
CE_scenario_path = scenario_base_path / 'circular_economy_scenarios'


In [ ]:
# Run simulations for all scenarios

model_runs = {} # to store outputs for all scenarios

# Loop over scenarios
for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running scenario: {climate_scen} from {CP_scenario_path}")
    print(f"Circular economy scenario: {circular_scen}")
    climate_policy_scenario_dir = CP_scenario_path / climate_scen           # path to climate policy scenario data (i.e., from IMAGE)
    circular_economy_scenario_dirs = {
        scenario: CE_scenario_path / scenario for scenario in circular_scen # path to circular economy scenario data
    }

# Get preprocessing data for all sectors and adjust material indices
# #REST OF
    rest_sector = get_preprocessing_data(sector="rest_of",base_dir=scenario_base_path,
                                         climate_policy_scenario_dir=climate_policy_scenario_dir,
                                         circular_economy_scenario_dirs=circular_economy_scenario_dirs,
                                         scenario_name=climate_scen)
    

# Build and run the model
    factory = ModelFactory(rest_sector,complete_timeline
                           ).add(RestOf)                

    model = factory.finish() # create the model
    model_runs[climate_scen] = model
    
    # Run the simulation (suppressing warnings)
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline) 
    
    print(f"Finished {scen_id}")


In [ ]:
model_runs.get('SSP2').rest_of.get('inflow_materials')